# Hierarchical clustering of mouse oral swabs

Figure 1a. Ward clustering and stacked family-level relative abundance profiles.

In [ ]:
from pathlib import Path

# Jupyter normally starts the kernel in the notebook directory.
here = Path.cwd()
repo_root = next(parent for parent in (here, *here.parents)
                 if (parent / "requirements.txt").is_file()
                 and (parent / "mouse").is_dir() and (parent / "human").is_dir())
data_dir = repo_root / "data"
output_dir = repo_root / "outputs" / 'mouse/01_dendrogram'
output_dir.mkdir(parents=True, exist_ok=True)
notebook_output = output_dir


## Load the abundance workbook and identify metadata

In [ ]:
import pandas as pd

# Define the file path
path = data_dir / 'Mouse_Data_For_PCA.xlsx'

# Read the Excel file into a DataFrame
df = pd.read_excel(path)
# Remove rows where all values are NaN
df = df.dropna(how='all')

df


In [ ]:
# Define label columns (metadata)
label_cols = ['Full ID', 'Group', 'Timepoint', 'Sex']

# Define feature columns (everything else)
feature_cols = [col for col in df.columns if col not in label_cols]

# Create separate DataFrames
df_labels = df[label_cols]
df_features = df[feature_cols]


## Cluster samples and plot composition

In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
import numpy as np
import pandas as pd

# -------------------------------
# Global font and style settings
# -------------------------------
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 14,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.titlesize': 20
})
plt.rcParams['pdf.fonttype'] = 42  # TrueType (works well with Inkscape)
plt.rcParams['ps.fonttype'] = 42

# -------------------------------
# 1. Composite sample label
# -------------------------------
df_labels['Label'] = df_labels['Sex'] + "_" + df_labels['Timepoint'] + "_" + df_labels['Group']

# -------------------------------
# 1.5. Keep up to top 16 families, group the rest into "Other", normalize to 1
# -------------------------------
df_feat_numeric = df_features.select_dtypes(include=[np.number])

mean_abundance = df_feat_numeric.mean().sort_values(ascending=False)
print(f"Found {len(mean_abundance)} bacterial families in dataset.")

top_n = min(16, len(mean_abundance) - 1) if len(mean_abundance) > 1 else len(mean_abundance)
top_families = mean_abundance.index[:top_n]
other_families = mean_abundance.index[top_n:]

if len(other_families) > 0:
    df_features_filtered = df_feat_numeric[top_families].copy()
    df_features_filtered['Other'] = df_feat_numeric[other_families].sum(axis=1)
else:
    df_features_filtered = df_feat_numeric[top_families].copy()

# Normalize rows to sum to 1 (100%)
df_features_filtered = df_features_filtered.div(df_features_filtered.sum(axis=1), axis=0)
print(f"Final plotted families: {len(df_features_filtered.columns)} --> {list(df_features_filtered.columns)}")

# -------------------------------
# 2. Hierarchical clustering
# -------------------------------
linkage_matrix = linkage(df_features_filtered, method='ward')

# -------------------------------
# 3. Figure setup
# -------------------------------
fig = plt.figure(figsize=(20, 15))
gs = fig.add_gridspec(5, 1, height_ratios=[1, 0.05, 0.05, 0.05, 2], hspace=0.05)
ax_dendro = fig.add_subplot(gs[0])
ax_strip_sex = fig.add_subplot(gs[1], sharex=ax_dendro)
ax_strip_time = fig.add_subplot(gs[2], sharex=ax_dendro)
ax_strip_group = fig.add_subplot(gs[3], sharex=ax_dendro)
ax_bar = fig.add_subplot(gs[4], sharex=ax_dendro)

# -------------------------------
# 4. Dendrogram
# -------------------------------
dendro = dendrogram(
    linkage_matrix,
    labels=df_labels['Label'].values,
    leaf_rotation=90,
    leaf_font_size=8,
    ax=ax_dendro,
    color_threshold=0,
    link_color_func=lambda k: "black"
)

ax_dendro.set_xticks([])
ax_dendro.set_xticklabels([])
ax_dendro.set_ylabel('Distance')
ax_dendro.set_title("Hierarchical Clustering of Bacterial Relative Abundance", fontweight='bold')

# -------------------------------
# 5. Reorder samples based on dendrogram leaves
# -------------------------------
ordered_idx = dendro['leaves']
ordered_features = df_features_filtered.iloc[ordered_idx]
ordered_labels = df_labels.iloc[ordered_idx]

# -------------------------------
# 6. Align bar chart to dendrogram width
# -------------------------------
x_start, x_end = ax_dendro.get_xlim()
x = np.linspace(x_start, x_end, len(ordered_idx))
bar_width = (x[1] - x[0]) * 0.8

for ax in [ax_strip_sex, ax_strip_time, ax_strip_group, ax_bar]:
    ax.set_xticks([])
    ax.set_xticklabels([])

# -------------------------------
# 7. Metadata strips (custom colors)
# -------------------------------
group_color_map = {"B6": "blue", "M83(ho)": "orange"}
sex_color_map = {"Male": "lightskyblue", "Female": "magenta"}
time_color_map = {"2 Months": "gold", "4 Months": "mediumpurple"}

group_colors = ordered_labels['Group'].map(group_color_map).fillna("lightgray")
sex_colors = ordered_labels['Sex'].map(sex_color_map).fillna("lightgray")
time_colors = ordered_labels['Timepoint'].map(time_color_map).fillna("lightgray")

ax_strip_group.bar(x, [1]*len(x), color=group_colors, width=bar_width, align='center')
ax_strip_sex.bar(x, [1]*len(x), color=sex_colors, width=bar_width, align='center')
ax_strip_time.bar(x, [1]*len(x), color=time_colors, width=bar_width, align='center')

for ax in [ax_strip_sex, ax_strip_time, ax_strip_group]:
    ax.set_yticks([])
    ax.set_ylim(0, 1.05)

# -------------------------------
# Add strip labels to the right
# -------------------------------
x_min, x_max = ax_dendro.get_xlim()
x_right = x_max + (x_max - x_min) * 0.02  # position a bit to the right

ax_strip_group.text(x_right, 0.5, 'Group', va='center', ha='left', fontsize=14, fontweight='bold')
ax_strip_sex.text(x_right, 0.5, 'Sex', va='center', ha='left', fontsize=14, fontweight='bold')
ax_strip_time.text(x_right, 0.5, 'Timepoint', va='center', ha='left', fontsize=14, fontweight='bold')

# -------------------------------
# 8. Stacked bar chart for bacterial families
# -------------------------------
bottoms = np.zeros(len(ordered_features))
colors = plt.cm.get_cmap('tab20', len(ordered_features.columns)).colors

for i, col in enumerate(ordered_features.columns):
    ax_bar.bar(
        x,
        ordered_features[col],
        bottom=bottoms,
        color=colors[i % len(colors)],
        width=bar_width,
        label=col,
        align='center'
    )
    bottoms += ordered_features[col]

ax_bar.set_ylabel("Relative Abundance", fontsize=16, fontweight='bold')

# -------------------------------
# 8.5. Expand x-limits to prevent clipping
# -------------------------------
x_min, x_max = ax_dendro.get_xlim()
expand = (x_max - x_min) / len(ordered_idx) / 2
for ax in [ax_strip_sex, ax_strip_time, ax_strip_group, ax_bar]:
    ax.set_xlim(x_min - expand, x_max + expand)

# -------------------------------
# 9. Legends (only bacterial families)
# -------------------------------
family_handles = [plt.Rectangle((0,0),1,1,color=colors[i % len(colors)]) for i in range(len(ordered_features.columns))]
fig.legend(
    family_handles, ordered_features.columns, title="Bacterial Family",
    loc='center right',
    bbox_to_anchor=(1.02, 0.4),
    fontsize=12,
    title_fontsize=14,
    frameon=False
)

output_path = notebook_output

plt.savefig(output_path / "Mouse_Dendro.pdf", format="pdf", bbox_inches="tight")

plt.show()
